# A prior of `sigma=1` means three different things

Here are three columns from one panel: yield in kilograms, spend in dollars, rainfall as a
z-score. Their scales differ by two orders of magnitude. Put a `Normal(0, 1)` prior on a
coefficient for each — the default any tutorial suggests — and you have declared three
completely different beliefs, none of which you hold.

That is why models are fit on scaled columns. The trap is the other half: a scaled column is
a number without units, and an analysis that scales on the way in and forgets on the way out
reports a lift of 0.42 with no statement of *0.42 of what*.

A scaled column is dimensionless — it is the column divided by a reference in the same unit.
`ScalingParameters` is a `Spec`, so the reference used travels with the analysis and
`unscale` restores the original dimension exactly.

In [ ]:
import numpy as np
import pandas as pd

from axiom.core import Covariate, D, Outcome, Treatment, dimensionless
from axiom.data import ColumnScaling, Panel, RoleMap, ScalingMethod, ScalingParameters, fit_scaling

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, scatter_fit

enable();  # every axiom result renders itself from here on

In [ ]:
rng = np.random.default_rng(11)
df = pd.DataFrame(
    {
        "u": np.repeat(["a", "b", "c"], 10),
        "t": np.tile(range(10), 3),
        "y": rng.gamma(4.0, 25.0, 30),
        "x": rng.uniform(0, 500, 30),
        "z": rng.normal(10, 3, 30),
    }
)
roles = RoleMap(
    unit="u", time="t",
    outcome=("y", Outcome(name="y_total", dimension=D.outcome)),
    treatments={"x": Treatment(name="x_dose", dimension=D.currency, unit="USD")},
    covariates={"z": Covariate(name="z", dimension=dimensionless())},
)
panel = Panel(df, roles)

In [ ]:
raw = {c: float(np.nanmax(np.abs(panel.column(c)))) for c in ("y", "x", "z")}
fig = compare(
    [f"{c}  (max {v:,.0f})" for c, v in raw.items()], list(raw.values()),
    value_fmt="{:,.0f}",
    title="Three columns, one prior, three different beliefs",
    subtitle="largest absolute value per column, before scaling",
    x_title="raw magnitude",
)
caption(fig, "A Normal(0, 1) coefficient prior against the middle column says something "
             "fifty times weaker than the same prior against the last one. Nothing in the "
             "model declaration shows it.")

`fit_scaling` chooses a `ScalingMethod` per role: by default treatments and the outcome are
scaled by their max (so 1.0 is "the largest dose seen") and covariates are standardized.
Those defaults are chosen so a coefficient means something a person can state out loud: with
a max-scaled dose, the coefficient is the effect of going from nothing to the largest dose
in the data.

In [ ]:
params: ScalingParameters = fit_scaling(panel)
table(
    [[col, cs.method, f"{cs.loc:.3f}", f"{cs.scale:.3f}"] for col, cs in params.columns.items()],
    headers=("column", "method", "loc", "scale"),
)

In [ ]:
scaled = params.scale(panel)
print(scaled.frame[["y", "x", "z"]].describe().loc[["mean", "std", "max"]].round(3))

In [ ]:
after = {c: float(np.nanmax(np.abs(scaled.column(c)))) for c in ("y", "x", "z")}
fig = compare(
    list(after), list(after.values()),
    value_fmt="{:.2f}",
    title="…and after",
    subtitle="the same three columns, on a scale a prior can be written against",
    x_title="scaled magnitude",
)
caption(fig, "Now one prior means one thing. The reference that got each column here is a "
             "Spec, so it is saved with the analysis rather than living in the script that "
             "happened to load the data.")

In [ ]:
back = params.unscale(scaled)
print("exact inverse:", np.allclose(back.frame["y"], panel.frame["y"]), np.allclose(back.frame["z"], panel.frame["z"]))
print("per column:", params.unscale_column("x", scaled.frame["x"].to_numpy()[:3]), panel.frame["x"].to_numpy()[:3])

In [ ]:
fig = scatter_fit(
    panel.frame["x"].to_numpy(), back.frame["x"].to_numpy(),
    title="Round trip, to the last bit",
    subtitle="original dose against scale-then-unscale — every point on the line, not near it",
    x_title="original (USD)", y_title="unscaled (USD)",
)
caption(fig, "This is the property that lets a fitted coefficient be reported in dollars. "
             "An approximate inverse would put a small, invisible, systematic error into "
             "every number that leaves the model.")

Methods are selectable per role, and a `ColumnScaling` can be built by hand when a reference
should come from outside the data (a published reference dose, for instance — review B9).
That case matters more than it sounds: scaling by *this* sample's maximum makes two studies
incomparable, because each one's 1.0 is its own.

In [ ]:
alt = fit_scaling(panel, treatments="mean", outcome="none", covariates="none")
print({c: s.method for c, s in alt.columns.items()})

ref: ScalingMethod = "max"
manual = ScalingParameters(columns={"x": ColumnScaling(method=ref, scale=1000.0)})
print(manual.scale(panel).frame["x"].max(), "<- x / 1000 USD reference")

Degenerate columns are refused rather than producing a divide-by-zero silently. A constant
column has no scale, and the honest response is to say so rather than to emit `inf` and let
it propagate into a log density.

In [ ]:
try:
    fit_scaling(Panel(df.assign(z=1.0), roles))
except ValueError as e:
    print("refused:", e)

In [ ]:
print(params.content_hash()[:16], "| round-trips:", ScalingParameters.from_json(params.to_json()) == params)

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
import pandas as pd

from axiom.core import D, Outcome
from axiom.data import Panel, RoleMap
from axiom.display import show
from axiom.viz import panel_coverage

frame = pd.DataFrame(
    {
        "unit": ["a"] * 5 + ["b"] * 4 + ["c"] * 5,
        "period": [1, 2, 3, 4, 5, 1, 2, 4, 5, 1, 2, 3, 4, 5],
        "y": [0.1] * 14,
    }
)
panel = Panel(frame, RoleMap(unit="unit", time="period", outcome=("y", Outcome(name="y", dimension=D.outcome))))
show(panel.completeness())
panel_coverage(panel)

Unbalance has a *pattern*. Unit `b` is missing period 3, and a count of missing rows would
report the gap without saying where it is.

## What this bought you

A model fit on numbers of comparable size, priors that mean what they say, and results that
come back in kilograms and dollars because the reference that took them out is stored beside
the posterior rather than remembered.